In [1]:
!python --version

Python 3.11.13


In [2]:
!pip install tqdm

In [ ]:
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu121

In [ ]:
!pip install torchvision

In [ ]:
import torch
torch.__version__

In [ ]:
!pip install pandas

In [ ]:
!pip install scikit-learn

In [ ]:
!pip install --upgrade albumentations

In [ ]:
!pip install matplotlib

# TRIESCH'S ALPHABETS

In [ ]:
import matplotlib.pyplot as plt
import torch
import torchvision

from torch import nn
from torchvision import transforms
from helper_functions import set_seeds

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
# 1. Get pretrained weights for ViT-Base
pretrained_vit_weights = torchvision.models.ViT_B_16_Weights.DEFAULT 

# 2. Setup a ViT model instance with pretrained weights
pretrained_vit = torchvision.models.vit_b_16(weights=pretrained_vit_weights).to(device)

# 3. Freeze the base parameters
for parameter in pretrained_vit.parameters():
    parameter.requires_grad = False
    
# 4. Change the classifier head 
class_names = ['A','B','C','Five','Point','V']

set_seeds()
pretrained_vit.heads = nn.Linear(in_features=768, out_features=len(class_names)).to(device)
# pretrained_vit # uncomment for model output 

In [ ]:
import torch
import torchvision

In [ ]:
!pip install torchinfo

In [ ]:
from torchinfo import summary

# Print a summary using torchinfo (uncomment for actual output)
summary(model=pretrained_vit, 
        input_size=(32, 3, 224, 224), # (batch_size, color_channels, height, width)
        # col_names=["input_size"], # uncomment for smaller output
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
)

In [ ]:
# Setup directory paths to train and test images
train_dir = r"C:\D\G\ISLAPHABETS\shp-marcel\shp_marcel_train\Marcel-Train"
test_dir = r"C:\D\G\ISLAPHABETS\shp-marcel\shp_marcel_test\Marcel-Test complex"

In [ ]:
# Get automatic transforms from pretrained ViT weights
pretrained_vit_transforms = pretrained_vit_weights.transforms()
print(pretrained_vit_transforms)

In [ ]:
import os

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

NUM_WORKERS = os.cpu_count()

def create_dataloaders(
    train_dir: str, 
    test_dir: str, 
    transform: transforms.Compose, 
    batch_size: int, 
    num_workers: int=NUM_WORKERS
):

  # Use ImageFolder to create dataset(s)
  train_data = datasets.ImageFolder(train_dir, transform=transform)
  test_data = datasets.ImageFolder(test_dir, transform=transform)

  # Get class names
  class_names = train_data.classes

  # Turn images into data loaders
  train_dataloader = DataLoader(
      train_data,
      batch_size=batch_size,
      shuffle=True,
      num_workers=num_workers,
      pin_memory=True,
  )
  test_dataloader = DataLoader(
      test_data,
      batch_size=batch_size,
      shuffle=False,
      num_workers=num_workers,
      pin_memory=True,
  )

  return train_dataloader, test_dataloader, class_names

In [ ]:
# Setup dataloaders
train_dataloader_pretrained, test_dataloader_pretrained, class_names = create_dataloaders(train_dir=train_dir,
                                                                                                     test_dir=test_dir,
                                                                                                     transform=pretrained_vit_transforms,
                                                                                                     batch_size=32) # Could increase if we had more samples, such as here: https://arxiv.org/abs/2205.01580 (there are other improvements there too...)

In [ ]:
from going_modular.going_modular import engine

# Create optimizer and loss function
optimizer = torch.optim.Adam(params=pretrained_vit.parameters(), 
                             lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

# Train the classifier head of the pretrained ViT feature extractor model
set_seeds()
pretrained_vit_results = engine.train(model=pretrained_vit,
                                      train_dataloader=train_dataloader_pretrained,
                                      test_dataloader=test_dataloader_pretrained,
                                      optimizer=optimizer,
                                      loss_fn=loss_fn,
                                      epochs=10,
                                      device=device)

In [ ]:
import os
import cv2
import torch
import torch.nn.functional as F
import time
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix

# Define the path to the folder containing test images
test_images_folder = r"D:\G\ISLAPHABETS\shp-marcel\shp_marcel_test\Marcel-Test"

# Ensure the folder path exists
if not os.path.exists(test_images_folder):
    print(f"Error: The folder '{test_images_folder}' does not exist.")
else:
    print(f"Testing images from folder: {test_images_folder}")

# Define the same transformation used during training
real_time_transform = pretrained_vit_weights.transforms()

# Function to test model on images and plot confusion matrix
def test_with_confusion_matrix(model, image_folder, device, class_names):
    model.eval()  # Set model to evaluation mode
    
    class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}

    correct_predictions = 0
    total_images = 0
    total_inference_time = 0.0

    y_true = []  # True labels
    y_pred = []  # Predicted labels

    # Loop through each class folder
    for class_folder in os.listdir(image_folder):
        class_path = os.path.join(image_folder, class_folder)

        if not os.path.isdir(class_path):
            continue  # Skip non-folder files

        image_files = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Testing {len(image_files)} images from class '{class_folder}'.")

        for image_file in image_files:
            image_path = os.path.join(class_path, image_file)
            
            try:
                # Load and transform image
                image = Image.open(image_path).convert("RGB")
                transformed_image = real_time_transform(image).unsqueeze(0).to(device)

                # Measure inference time
                start_time = time.time()
                with torch.no_grad():
                    outputs = model(transformed_image)
                    probabilities = F.softmax(outputs, dim=1)  # Convert logits to probabilities
                    confidence, predicted = torch.max(probabilities, 1)  # Get highest probability and class index
                end_time = time.time()

                inference_time = end_time - start_time
                total_inference_time += inference_time

                predicted_class = class_names[predicted.item()]
                confidence_score = confidence.item() * 100  # Convert to percentage

                # Store true and predicted labels
                if class_folder in class_to_idx:
                    y_true.append(class_to_idx[class_folder])  # True label
                    y_pred.append(predicted.item())  # Predicted label

                # Compare prediction with actual class
                actual_class = class_folder
                is_correct = (predicted_class == actual_class)

                if is_correct:
                    correct_predictions += 1
                
                total_images += 1

                # Print result with confidence score and inference time
                print(f"Image: {image_file} --> Predicted: {predicted_class} ({confidence_score:.2f}%), "
                      f"Actual: {actual_class}, Correct: {is_correct}, Inference Time: {inference_time:.4f} sec")

                # Display image with prediction and confidence score
                img = cv2.imread(image_path)
                img = cv2.resize(img, (400, 400))  # Resize for better visualization

                label_text = f"{predicted_class} ({confidence_score:.1f}%)"
                cv2.putText(img, label_text, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                cv2.putText(img, f"{inference_time:.3f}s", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

                cv2.imshow("Prediction", img)
                cv2.waitKey(500)  # Display each image for 0.5 seconds

            except Exception as e:
                print(f"Error processing image {image_file}: {e}")

    cv2.destroyAllWindows()

    # Compute accuracy
    if total_images > 0:
        accuracy = (correct_predictions / total_images) * 100
        avg_inference_time = total_inference_time / total_images
        print(f"\nTest Accuracy: {accuracy:.2f}% ({correct_predictions}/{total_images} correct)")
        print(f"Average Inference Time per Image: {avg_inference_time:.4f} sec")

        # Compute Precision, Recall, and F1-score
        print("\nClassification Report:\n")
        print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

        # Compute confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", xticklabels=class_names, yticklabels=class_names)
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title("Confusion Matrix")
        plt.show()

    else:
        print("\nNo images found for testing.")

# Run testing on folder images and plot confusion matrix
test_with_confusion_matrix(pretrained_vit, test_images_folder, device, class_names)